# Hardware Model Comparison — Cross-Model Benchmark

Cross-model comparison of **latency**, **throughput**, **power**, and **energy efficiency**
across architectures, lambda values, seeds, and activation functions.

| Dimension | Values |
|---|---|
| **Architecture** | ResSHyp (residual+hyper), SHyp (no residual+hyper), ResFP (residual, no hyper), FP (no residual, no hyper) |
| **Lambda** | 1, 1000 (rate-distortion trade-off) |
| **Platforms** | GPU (RTX A4000), CPU (x86), FPGA (ZCU102) |

**Data sources:** JSON files in `results/benchmark/<model_name>/` produced by `run_full_benchmark.py`.

**Structure:**
1. Setup & Configuration
2. Load & Parse All Benchmark Results → unified `df_all`
3. Overview Table — all models × platforms
4. Latency vs Lambda (line charts per scenario)
5. Architecture Comparison (ResSHyp vs SHyp grouped bars)
6. Hardware Cost Scatter (quality vs latency / energy)
7. Export Summary CSV


## 1 · Setup & Configuration

In [ ]:
import json
import re
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT_DIR = Path("..").resolve()
BENCHMARK_ROOT = ROOT_DIR / "results" / "benchmark"

assert BENCHMARK_ROOT.exists(), f"Benchmark root not found: {BENCHMARK_ROOT}"
print(f"Benchmark root: {BENCHMARK_ROOT}")
print(f"Model dirs found: {[p.name for p in sorted(BENCHMARK_ROOT.iterdir()) if p.is_dir()]}")

# ── Load shared color palette (Okabe-Ito, colorblind-safe) ────────────────────
C = json.load(open(ROOT_DIR / "notebooks" / "plots_colors.json"))

# ── Plotting constants ─────────────────────────────────────────────────────────
PLATFORM_ORDER = ["gpu", "cpu", "fpga"]
PLATFORM_LABELS = {"gpu": "GPU (RTX A4000)", "cpu": "CPU (Intel Xeon)", "fpga": "FPGA (ZCU102)"}
PLATFORM_COLORS = {
    "gpu": C["platforms"]["gpu_dynamic"],
    "cpu": C["platforms"]["cpu_dynamic"],
    "fpga": C["platforms"]["fpga_dynamic"],
}
ARCHITECTURE_COLORS = {
    "ResSHyp": C["architectures"]["ResSHyp"],
    "SHyp": C["architectures"]["SHyp"],
    "ResFP": C["architectures"]["ResFP"],
    "FP": C["architectures"]["FP"],
}
SCENARIO_ORDER = ["full", "compress", "decompress", "nn_only", "entropy_only"]
SCENARIO_LABELS = {
    "full": "Full",
    "compress": "Compress",
    "decompress": "Decompress",
    "nn_only": "NN Only",
    "entropy_only": "Entropy Only",
}

SAVE_FIGURES = True
FIGURE_FORMAT = "pdf"
PLOTS_DIR = ROOT_DIR / "results" / "plots" / "hardware_model_comparison"

# ── Step-breakdown colours & order (mirrors benchmark_analysis.ipynb) ─────────
STEP_COLORS = {
    "nn_g_a": C["latency_steps"]["nn_g_a"],
    "nn_h_a": C["latency_steps"]["nn_h_a"],
    "nn_h_s": C["latency_steps"]["nn_h_s"],
    "nn_g_s": C["latency_steps"]["nn_g_s"],
    "cpu_eb_compress": C["latency_steps"]["cpu_eb_compress"],
    "cpu_eb_decompress": C["latency_steps"]["cpu_eb_decompress"],
    "cpu_gc_compress": C["latency_steps"]["cpu_gc_compress"],
    "cpu_gc_decompress": C["latency_steps"]["cpu_gc_decompress"],
    "cpu_concat_abs": C["latency_steps"]["cpu_concat_abs"],
    "cpu_split_y_hat": C["latency_steps"]["cpu_split_y_hat"],
}
STEP_ORDER = [
    "nn_g_a",
    "cpu_concat_abs",
    "nn_h_a",
    "cpu_eb_compress",
    "cpu_eb_decompress",
    "nn_h_s",
    "cpu_gc_compress",
    "cpu_gc_decompress",
    "cpu_split_y_hat",
    "nn_g_s",
]

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "font.size": 10,
    }
)


## 2 · Load & Parse All Benchmark Results

Parse model directory names using regex: `<arch>-<act>_s<seed>_L<lambda>_pt`.
Load all `benchmark_<platform>_<scenario>.json` files, extract latency + power metrics,
and build a unified `df_all` DataFrame — one row per `(model, platform, scenario)`.

In [ ]:
# ── Step-breakdown extraction ─────────────────────────────────────────────────
def _extract_step_breakdown(data: dict, platform: str) -> Dict[str, float]:
    """Return {step_label: mean_ms} from latency_breakdown, normalising prefixes
    to canonical 'nn_' and 'cpu_' for uniform cross-platform comparison.
    """
    breakdown = data.get("latency_breakdown", {})
    out: Dict[str, float] = {}
    for step, stats in breakdown.items():
        if not isinstance(stats, dict):
            continue
        mean_s = stats.get("mean_s", 0.0)
        canonical = step
        if platform == "gpu" and step.startswith("gpu_"):
            canonical = "nn_" + step[4:]
        elif platform == "fpga" and step.startswith("dpu_"):
            canonical = "nn_" + step[4:]
        out[canonical] = mean_s * 1000  # → ms
    return out


# ── Model name parser ──────────────────────────────────────────────────────────
_MODEL_RE = re.compile(r"^(?P<arch>[^-]+)-(?P<act>[^_]+)_s(?P<seed>\d+)_L(?P<lmbda>\d+)_pt$")


def parse_model_name(name: str) -> Optional[Dict[str, Any]]:
    """Parse model directory name → dict with arch, act, seed, lmbda."""
    m = _MODEL_RE.match(name)
    if not m:
        return None
    return {
        "architecture": m.group("arch"),
        "activation": m.group("act"),
        "seed": int(m.group("seed")),
        "lmbda": int(m.group("lmbda")),
    }


# ── Power extraction helpers (mirrors benchmark_analysis.ipynb) ───────────────
_FPGA_PL_RAILS = frozenset({"VCCINT", "VCCBRAM", "VCCAUX", "VCC1V2", "VCC3V3"})
_FPGA_PS_RAILS = frozenset(
    {
        "VCCPSINTFP",
        "VCCPSINTLP",
        "VCCPSAUX",
        "VCCPSPLL",
        "VCCO_PSDDR_504",
        "VCCOPS",
        "VCCOPS3",
        "VCCPSDDRPLL",
    }
)
_FPGA_MGT_RAILS = frozenset({"MGTAVCC", "MGTAVTT", "MGTRAVCC", "MGTRAVTT"})
_FPGA_FMC_RAILS = frozenset({"VADJ_FMC"})


def _classify_file(name: str) -> Tuple[str, str]:
    """Return (platform, scenario) from a benchmark JSON filename."""
    stem = name.replace(".json", "")
    parts = stem.split("_")
    if len(parts) >= 3 and parts[1] in ("gpu", "cpu", "fpga"):
        platform = parts[1]
        scenario = "_".join(parts[2:])
    else:
        platform = "fpga"
        scenario = "_".join(parts[1:])
    return platform, scenario


def _extract_power(data: dict, platform: str) -> Dict[str, Optional[float]]:
    """Flat power metrics — same logic as benchmark_analysis.ipynb."""
    pw = data.get("power", {})
    out: Dict[str, Optional[float]] = {
        "power_total_w": None,
        "power_nn_w": None,
        "power_cpu_w": None,
        "power_idle_total_w": None,
        "energy_total_j": None,
        "power_gpu_w": None,
        "power_cpu_rapl_w": None,
        "idle_gpu_w": None,
        "idle_cpu_rapl_w": None,
    }
    if not pw:
        return out

    if platform == "fpga":
        groups = pw.get("groups_avg_w", {})
        _tbp = groups.get("TBP") or pw.get("board_total_extended_avg_w")
        _fmc = groups.get("FMC", 0.0) or 0.0
        out["power_total_w"] = round(_tbp + _fmc, 4) if _tbp is not None else None
        out["power_nn_w"] = groups.get("DPU_fabric")
        out["power_cpu_w"] = groups.get("PS_compute")
        duration = next(
            (
                v["duration_s"]
                for v in pw.get("per_rail", {}).values()
                if isinstance(v, dict) and "duration_s" in v
            ),
            None,
        )
        if out["power_total_w"] is not None and duration:
            out["energy_total_j"] = out["power_total_w"] * duration
        idle = pw.get("idle_baseline", {})
        if idle:
            idle_ina226 = pw.get("idle_board_total_avg_w") or sum(
                v.get("avg_power_w", 0.0) for v in idle.values() if isinstance(v, dict)
            )
            out["power_idle_total_w"] = round(idle_ina226, 4)

    elif platform == "gpu":
        gpu_pw = pw.get("gpu", {})
        rapl_total = pw.get("cpu_rapl_total_avg_w") or 0.0
        gpu_w = gpu_pw.get("avg_power_w", 0.0) if gpu_pw else 0.0
        out["power_gpu_w"] = gpu_w if gpu_pw else None
        out["power_cpu_rapl_w"] = rapl_total if rapl_total else None
        out["power_nn_w"] = gpu_w
        out["power_cpu_w"] = rapl_total if rapl_total else None
        out["power_total_w"] = gpu_w + (rapl_total or 0.0)
        gpu_e = gpu_pw.get("energy_j", 0.0) if gpu_pw else 0.0
        rapl_e = sum(
            v.get("energy_j", 0.0) for v in pw.get("cpu_rapl", {}).values() if isinstance(v, dict)
        )
        out["energy_total_j"] = gpu_e + rapl_e
        idle_gpu = pw.get("idle_gpu", {})
        idle_rapl = pw.get("idle_cpu_rapl", {})
        idle_g = idle_gpu.get("avg_power_w", 0.0) if idle_gpu else 0.0
        idle_r = sum(v.get("avg_power_w", 0.0) for v in idle_rapl.values() if isinstance(v, dict))
        out["idle_gpu_w"] = idle_g if idle_gpu else None
        out["idle_cpu_rapl_w"] = idle_r if idle_rapl else None
        if idle_gpu or idle_rapl:
            out["power_idle_total_w"] = idle_g + idle_r

    elif platform == "cpu":
        rapl_total = pw.get("cpu_rapl_total_avg_w") or 0.0
        out["power_total_w"] = rapl_total if rapl_total else None
        out["power_cpu_rapl_w"] = rapl_total if rapl_total else None
        rapl_e = sum(
            v.get("energy_j", 0.0) for v in pw.get("cpu_rapl", {}).values() if isinstance(v, dict)
        )
        out["energy_total_j"] = rapl_e if rapl_e else None
        idle_rapl = pw.get("idle_cpu_rapl", {})
        if idle_rapl:
            idle_r = sum(
                v.get("avg_power_w", 0.0) for v in idle_rapl.values() if isinstance(v, dict)
            )
            out["idle_cpu_rapl_w"] = idle_r
            out["power_idle_total_w"] = idle_r

    return out


# ── Main data loader ───────────────────────────────────────────────────────────
def load_all_models(benchmark_root: Path) -> pd.DataFrame:
    """Scan all model directories, parse names, load JSONs → unified DataFrame."""
    rows: List[Dict[str, Any]] = []
    skipped: List[str] = []

    for model_dir in sorted(benchmark_root.iterdir()):
        if not model_dir.is_dir():
            continue
        parsed = parse_model_name(model_dir.name)
        if parsed is None:
            skipped.append(model_dir.name)
            continue

        for jpath in sorted(model_dir.glob("benchmark_*.json")):
            platform, scenario = _classify_file(jpath.name)
            if scenario.endswith("_sequential"):
                continue  # exclude sequential variants
            with open(jpath) as f:
                data: dict = json.load(f)

            latency_ms = data.get("latency_total_mean_ms", 0.0)
            throughput_fps = data.get("throughput_fps", 0.0)
            nn_key = next(
                (
                    k
                    for k in data
                    if k.startswith("latency_")
                    and k.endswith("_total_mean_ms")
                    and k not in ("latency_total_mean_ms", "latency_cpu_total_mean_ms")
                ),
                None,
            )
            nn_latency_ms = data.get(nn_key, 0.0) if nn_key else 0.0
            cpu_latency_ms = data.get("latency_cpu_total_mean_ms", 0.0)
            avg_bytes = data.get("avg_compressed_bytes")
            bpp = (avg_bytes * 8 / (256 * 256)) if avg_bytes else None
            power = _extract_power(data, platform)
            steps = _extract_step_breakdown(data, platform)

            energy_per_tile_mj = None
            if power["power_total_w"] is not None and latency_ms > 0:
                energy_per_tile_mj = power["power_total_w"] * (latency_ms / 1000) * 1000

            row: Dict[str, Any] = {
                "model_name": model_dir.name,
                **parsed,
                "platform": platform,
                "scenario": scenario,
                "latency_ms": latency_ms,
                "nn_latency_ms": nn_latency_ms,
                "cpu_latency_ms": cpu_latency_ms,
                "nn_fraction": nn_latency_ms / latency_ms if latency_ms > 0 else 0.0,
                "throughput_fps": throughput_fps,
                "bpp": bpp,
                "energy_per_tile_mj": energy_per_tile_mj,
                "_step_breakdown": steps,
                **power,
            }
            rows.append(row)

    if skipped:
        print(f"  ⚠ Skipped unrecognised dirs: {skipped}")

    df = pd.DataFrame(rows)
    df["platform"] = pd.Categorical(df["platform"], categories=PLATFORM_ORDER, ordered=True)
    df["scenario"] = pd.Categorical(df["scenario"], categories=SCENARIO_ORDER, ordered=True)
    df["architecture"] = pd.Categorical(
        df["architecture"], categories=list(ARCHITECTURE_COLORS), ordered=True
    )
    df = df.sort_values(["model_name", "platform", "scenario"]).reset_index(drop=True)
    # Derived energy column in Joules
    df["energy_per_tile_j"] = df["energy_per_tile_mj"] / 1000.0
    return df


df_all = load_all_models(BENCHMARK_ROOT)
print(f"\nLoaded {len(df_all)} rows from {df_all['model_name'].nunique()} model(s).")
print(
    df_all.groupby(["architecture", "lmbda", "seed", "platform"])["scenario"]
    .count()
    .rename("n_scenarios")
    .to_string()
)


## 3 · Overview Table — All Models × Platforms

Pivot table: mean latency (ms) for the `full` scenario, indexed by (architecture, λ), columns = platform.

In [ ]:
full = df_all[df_all["scenario"] == "full"].copy()

# ── Latency pivot ──────────────────────────────────────────────────────────────
pivot_lat = full.pivot_table(
    index=["architecture", "lmbda"],
    columns="platform",
    values="latency_ms",
    aggfunc="mean",
)
print("Mean latency (ms) — full scenario")
display(
    pivot_lat.style.format("{:.2f}", na_rep="—")
    .highlight_min(axis=1, props="background-color:#d4edda; font-weight:bold")
    .set_caption("Latency (ms) — full scenario — mean across seeds")
)

# ── Energy pivot ───────────────────────────────────────────────────────────────
pivot_eng = full.pivot_table(
    index=["architecture", "lmbda"],
    columns="platform",
    values="energy_per_tile_mj",
    aggfunc="mean",
)
print("\nMean energy per tile (mJ) — full scenario")
display(
    pivot_eng.style.format("{:.1f}", na_rep="—")
    .highlight_min(axis=1, props="background-color:#d4edda; font-weight:bold")
    .set_caption("Energy per tile (mJ) — full scenario — mean across seeds")
)

# ── Coverage check ─────────────────────────────────────────────────────────────
expected = {(p, s) for p in PLATFORM_ORDER for s in SCENARIO_ORDER}
for model in df_all["model_name"].unique():
    sub = df_all[df_all["model_name"] == model]
    actual = set(zip(sub["platform"].astype(str), sub["scenario"].astype(str)))
    missing = expected - actual
    if missing:
        print(f"\n⚠ {model} — missing {len(missing)} combinations: {sorted(missing)}")
    else:
        print(f"✓ {model} — all {len(expected)} (platform × scenario) combinations present.")


## 4 · Latency vs Lambda

Line chart: **x = λ**, **y = latency (ms)**, one line per `(architecture, platform)`.

Key hypothesis:
- `full` and `entropy_only` latencies **increase with λ** (higher λ → larger latent space → more entropy coding symbols).
- `nn_only` latency is **flat** (λ controls rate-distortion trade-off, not NN architecture).

In [ ]:
PLAT_LINESTYLES = {"gpu": "-", "cpu": "--", "fpga": "-."}
PLAT_MARKERS = {"gpu": "o", "cpu": "s", "fpga": "^"}


def plot_latency_vs_lambda(
    df: pd.DataFrame,
    scenarios: Optional[List[str]] = None,
) -> None:
    """Line chart: latency vs lambda, one line per (architecture, platform).

    x-axis: lambda (log scale)
    y-axis: mean latency (ms) across seeds
    Error band: min/max across seeds
    """
    scenarios = scenarios or ["full", "compress", "decompress", "nn_only", "entropy_only"]
    # Only keep scenarios that have data
    scenarios = [
        s for s in scenarios if s in df["scenario"].cat.categories and (df["scenario"] == s).any()
    ]
    if not scenarios:
        print("No data for any of the requested scenarios.")
        return

    lambdas = sorted(df["lmbda"].unique())
    archs = [a for a in ARCHITECTURE_COLORS if a in df["architecture"].values]
    plats = [p for p in PLATFORM_ORDER if p in df["platform"].cat.categories]

    n_sc = len(scenarios)
    ncols = min(3, n_sc)
    nrows = (n_sc + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows), squeeze=False)
    axes_flat = axes.flatten()

    for ax_idx, sc in enumerate(scenarios):
        ax = axes_flat[ax_idx]
        sub = df[df["scenario"] == sc]

        for arch in archs:
            for plat in plats:
                grp = sub[(sub["architecture"] == arch) & (sub["platform"] == plat)]
                if grp.empty:
                    continue
                stats = grp.groupby("lmbda")["latency_ms"].agg(["mean", "min", "max"])
                if stats.empty or len(stats) < 2:
                    # Single lambda: plot as a scatter point only
                    ax.scatter(
                        stats.index,
                        stats["mean"],
                        color=ARCHITECTURE_COLORS[arch],
                        marker=PLAT_MARKERS.get(plat, "o"),
                        s=60,
                        zorder=5,
                        label=f"{arch} / {PLATFORM_LABELS.get(plat, plat)}",
                    )
                    continue
                ax.plot(
                    stats.index,
                    stats["mean"],
                    color=ARCHITECTURE_COLORS[arch],
                    linestyle=PLAT_LINESTYLES.get(plat, "-"),
                    marker=PLAT_MARKERS.get(plat, "o"),
                    linewidth=1.8,
                    markersize=6,
                    label=f"{arch} / {PLATFORM_LABELS.get(plat, plat)}",
                )
                if len(grp["seed"].unique()) > 1:
                    ax.fill_between(
                        stats.index,
                        stats["min"],
                        stats["max"],
                        color=ARCHITECTURE_COLORS[arch],
                        alpha=0.15,
                    )

        ax.set_xscale("log")
        ax.set_xticks(lambdas)
        ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
        ax.set_xlabel("λ (rate-distortion weight)")
        ax.set_ylabel("Latency (ms)")
        ax.set_title(SCENARIO_LABELS.get(sc, sc))
        ax.legend(fontsize=7, loc="upper left", ncol=1)

    # Hide unused axes
    for j in range(len(scenarios), len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle("Latency vs λ — per scenario and platform", fontsize=13, y=1.01)
    fig.tight_layout()
    if SAVE_FIGURES:
        PLOTS_DIR.mkdir(parents=True, exist_ok=True)
        plt.savefig(PLOTS_DIR / f"latency_vs_lambda.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()


plot_latency_vs_lambda(df_all)


## 5 · Architecture Comparison — ResSHyp / SHyp / ResFP / FP

Grouped bar chart: **x = platform**, bars grouped by architecture, one subplot per scenario.
Annotates speedup ratio of the second arch relative to the first above each pair.

Architectures are included automatically if data is present (`ARCHITECTURE_COLORS` order: ResSHyp → SHyp → ResFP → FP).

> ℹ️ Architectures without benchmark results are silently skipped.


In [ ]:
def plot_architecture_comparison(
    df: pd.DataFrame,
    metric: str = "latency_ms",
    ylabel: str = "Latency (ms)",
    scenarios: Optional[List[str]] = None,
    lmbda_filter: Optional[int] = None,
    value_fmt: str = "{:.1f}",
) -> None:
    """Grouped bar chart comparing architectures per platform and scenario.

    Parameters
    ----------
    df           : df_all or filtered subset
    metric       : Column to plot (e.g. 'latency_ms', 'energy_per_tile_j')
    ylabel       : Y-axis label
    scenarios    : List of scenarios to show (default: full, compress, decompress, nn_only)
    lmbda_filter : If set, restrict to this lambda value (recommended to pick one)
    value_fmt    : Format string for bar value labels (e.g. '{:.1f}' or '{:.3f}')
    """
    scenarios = scenarios or ["full", "compress", "decompress", "nn_only"]
    if lmbda_filter is not None:
        df = df[df["lmbda"] == lmbda_filter]

    archs = [a for a in ARCHITECTURE_COLORS if a in df["architecture"].values]
    plats = [
        p
        for p in PLATFORM_ORDER
        if p in df["platform"].cat.categories and (df["platform"] == p).any()
    ]

    if not archs:
        print("No architecture data available.")
        return
    if len(archs) < 2:
        print(
            f"Only one architecture present ({archs[0]}); skipping comparison bars. "
            "Add SHyp benchmark results to enable this plot."
        )

    n_sc = len(scenarios)
    ncols = min(2, n_sc)
    nrows = (n_sc + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4.5 * nrows), squeeze=False)
    axes_flat = axes.flatten()

    x = np.arange(len(plats))
    n_arch = len(archs)
    bar_w = 0.7 / max(n_arch, 1)

    for ax_idx, sc in enumerate(scenarios):
        ax = axes_flat[ax_idx]
        sub = df[df["scenario"] == sc]

        arch_means: Dict[str, List[float]] = {}
        for arch in archs:
            vals = []
            for plat in plats:
                grp = sub[(sub["architecture"] == arch) & (sub["platform"] == plat)]
                vals.append(float(grp[metric].mean()) if not grp.empty else 0.0)
            arch_means[arch] = vals

            offset = (archs.index(arch) - n_arch / 2 + 0.5) * bar_w
            bars = ax.bar(
                x + offset,
                vals,
                bar_w * 0.88,
                color=ARCHITECTURE_COLORS[arch],
                label=arch,
                edgecolor="white",
                linewidth=0.6,
            )
            for bar, v in zip(bars, vals):
                if v > 0:
                    ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        bar.get_height(),
                        value_fmt.format(v),
                        ha="center",
                        va="bottom",
                        fontsize=7.5,
                    )

        ax.set_xticks(x)
        ax.set_xticklabels([PLATFORM_LABELS.get(p, p) for p in plats], fontsize=8, rotation=10)
        ax.set_ylabel(ylabel)
        ax.set_title(SCENARIO_LABELS.get(sc, sc))
        ax.legend(fontsize=8, loc="upper right")

    for j in range(len(scenarios), len(axes_flat)):
        axes_flat[j].set_visible(False)

    lmbda_str = f" — λ={lmbda_filter}" if lmbda_filter is not None else ""
    fig.suptitle(f"Architecture Comparison: {ylabel}{lmbda_str}", fontsize=13, y=1.01)
    fig.tight_layout()
    if SAVE_FIGURES:
        PLOTS_DIR.mkdir(parents=True, exist_ok=True)
        safe_metric = metric.replace("/", "_")
        plt.savefig(
            PLOTS_DIR / f"architecture_comparison_{safe_metric}.{FIGURE_FORMAT}",
            bbox_inches="tight",
        )
    plt.show()


# # ── Latency comparison ─────────────────────────────────────────────────────────
# for lmbda_val in sorted(df_all["lmbda"].unique()):
#     print(f"\n── λ = {lmbda_val} ────────────────────────────────────────────────")
#     plot_architecture_comparison(df_all, "latency_ms", "Latency (ms)", lmbda_filter=lmbda_val)
#     plot_architecture_comparison(df_all, "energy_per_tile_j", "Energy per tile (J)", lmbda_filter=lmbda_val, value_fmt="{:.3f}")

plot_architecture_comparison(df_all, "latency_ms", "Latency (ms)", lmbda_filter=1000)
plot_architecture_comparison(
    df_all, "energy_per_tile_j", "Energy per tile (J)", lmbda_filter=1000, value_fmt="{:.3f}"
)


## 5b · Latency Step Breakdown — Cross-Architecture

Stacked horizontal bar chart showing how total latency is composed of individual pipeline
steps (`nn_g_a`, `nn_h_a`, `nn_h_s`, `nn_g_s`, `cpu_*`).

**Layout:** one subplot per platform. Each bar = one architecture (mean across seeds).
Filter by a single `scenario` and `lmbda` to keep the comparison apples-to-apples.

> Steps smaller than `min_ms` (default 0.1 ms) are hidden to reduce clutter.


In [ ]:
def plot_step_breakdown_comparison(
    df: pd.DataFrame,
    scenario: str = "full",
    lmbda_filter: Optional[int] = None,
    min_ms: float = 0.1,
) -> None:
    """Stacked horizontal bar chart: latency step breakdown per architecture.

    One subplot per platform.  Each bar = one architecture, averaged across seeds.
    Segments correspond to individual pipeline steps (NN subgraphs + CPU ops).

    Parameters
    ----------
    df           : df_all or filtered subset
    scenario     : Scenario to plot (default: 'full')
    lmbda_filter : If set, restrict to this lambda value
    min_ms       : Steps smaller than this (ms) are hidden
    """
    sub = df[df["scenario"] == scenario].copy()
    if lmbda_filter is not None:
        sub = sub[sub["lmbda"] == lmbda_filter]
    if sub.empty:
        print(
            f"No data for scenario='{scenario}'"
            + (f", lmbda={lmbda_filter}" if lmbda_filter else "")
            + "."
        )
        return

    archs = [a for a in ARCHITECTURE_COLORS if a in sub["architecture"].values]
    plats = [
        p
        for p in PLATFORM_ORDER
        if p in sub["platform"].cat.categories and (sub["platform"] == p).any()
    ]

    if not archs or not plats:
        print("No data available.")
        return

    ncols = len(plats)
    fig, axes = plt.subplots(
        1,
        ncols,
        figsize=(7 * ncols, max(2.5, len(archs) * 0.9 + 1.5)),
        sharey=False,
        squeeze=False,
    )

    for col_idx, plat in enumerate(plats):
        ax = axes[0][col_idx]
        plat_sub = sub[sub["platform"] == plat]

        y_labels: List[str] = []
        for row_idx, arch in enumerate(archs):
            arch_grp = plat_sub[plat_sub["architecture"] == arch]
            if arch_grp.empty:
                y_labels.append(arch)
                continue

            # Average step breakdown across seeds
            all_steps: Dict[str, List[float]] = {}
            for _, row in arch_grp.iterrows():
                for step, val in row["_step_breakdown"].items():
                    all_steps.setdefault(step, []).append(val)
            mean_steps = {s: float(np.mean(v)) for s, v in all_steps.items()}

            total_ms = arch_grp["latency_ms"].mean()
            y_labels.append(arch)

            # Order steps; skip tiny ones
            ordered = [
                (s, mean_steps.get(s, 0.0)) for s in STEP_ORDER if mean_steps.get(s, 0.0) >= min_ms
            ]
            # Append any steps not in STEP_ORDER
            known = set(STEP_ORDER)
            for s, v in sorted(mean_steps.items()):
                if s not in known and v >= min_ms and not s.startswith("_"):
                    ordered.append((s, v))

            left = 0.0
            for step_name, ms_val in ordered:
                color = STEP_COLORS.get(step_name, "#bdc3c7")
                ax.barh(
                    row_idx,
                    ms_val,
                    left=left,
                    height=0.6,
                    color=color,
                    edgecolor="white",
                    linewidth=0.5,
                    label=step_name,
                )
                if total_ms > 0 and ms_val > total_ms * 0.05:
                    ax.text(
                        left + ms_val / 2,
                        row_idx,
                        f"{ms_val:.1f}",
                        ha="center",
                        va="center",
                        fontsize=6.5,
                        color="white",
                        fontweight="bold",
                    )
                left += ms_val

        ax.set_yticks(range(len(archs)))
        ax.set_yticklabels(y_labels)
        ax.invert_yaxis()
        ax.set_xlabel("Latency (ms)")
        ax.set_title(PLATFORM_LABELS.get(plat, plat))

        # Deduplicated legend ordered by STEP_ORDER
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ordered_keys = [k for k in STEP_ORDER if k in by_label]
        ordered_keys += [k for k in by_label if k not in set(STEP_ORDER)]
        ax.legend(
            [by_label[k] for k in ordered_keys],
            ordered_keys,
            loc="lower right",
            fontsize=7,
            ncol=2,
        )

    lmbda_str = f" — λ={lmbda_filter}" if lmbda_filter is not None else ""
    fig.suptitle(
        f"Latency Step Breakdown — {SCENARIO_LABELS.get(scenario, scenario)}{lmbda_str}"
        " (mean across seeds)",
        fontsize=13,
        y=1.02,
    )
    fig.tight_layout()
    if SAVE_FIGURES:
        PLOTS_DIR.mkdir(parents=True, exist_ok=True)
        safe = f"{scenario}" + (f"_L{lmbda_filter}" if lmbda_filter else "")
        plt.savefig(PLOTS_DIR / f"step_breakdown_{safe}.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()


# ── Plot for the most informative scenario/lambda combinations ─────────────────
plot_step_breakdown_comparison(df_all, scenario="full", lmbda_filter=1000)
plot_step_breakdown_comparison(df_all, scenario="compress", lmbda_filter=1000)
plot_step_breakdown_comparison(df_all, scenario="decompress", lmbda_filter=1000)
plot_step_breakdown_comparison(df_all, scenario="nn_only", lmbda_filter=1000)
plot_step_breakdown_comparison(df_all, scenario="entropy_only", lmbda_filter=1000)


## 6 · Hardware Cost Scatter — Quality vs Latency / Energy

Scatter plot: **x = PSNR** (quality metric), **y = latency or energy**.

- Color by **platform** (`PLATFORM_COLORS`)
- Marker shape by **architecture**
- Label each point with its **λ** value
- Point **size ∝ BPP** (`s = bpp * 500`)

> ℹ️ PSNR values must be loaded from `results/fpga/` evaluation output or a separate metrics CSV.
> This cell loads from `data/method_ground_truths/` or skips gracefully if unavailable.

Two subplots: **left = latency**, **right = energy**.

In [ ]:
# ── Load per-model quality metrics ────────────────────────────────────────────
# Expected source: results/fpga/<model_name>/results/metrics_summary.json
# with at least {"psnr_mean": float, "bpp_mean": float} per model.
# Falls back to BPP from benchmark JSON (no PSNR) if not available.

ARCH_MARKERS = {"ResSHyp": "o", "SHyp": "s", "ResFP": "^", "FP": "D"}


def _load_model_quality(model_name: str) -> Dict[str, Optional[float]]:
    """Try to load PSNR / BPP from compiled model results."""
    candidates = [
        ROOT_DIR
        / "results"
        / "fpga"
        / "compiled_models"
        / model_name
        / "results"
        / "metrics_summary.json",
        ROOT_DIR / "results" / "fpga" / model_name / "metrics_summary.json",
    ]
    for path in candidates:
        if path.exists():
            d = json.load(open(path))
            return {"psnr": d.get("psnr_mean"), "bpp": d.get("bpp_mean")}
    return {"psnr": None, "bpp": None}


def plot_hardware_cost_scatter(
    df: pd.DataFrame,
    scenario: str = "full",
) -> None:
    """Scatter: x=PSNR, y=latency (left) or energy (right).

    One point per (model_name, platform).
    Color by platform, marker by architecture, size by BPP.
    """
    sub = df[df["scenario"] == scenario].copy()

    # Attach quality metrics
    quality_cache: Dict[str, Dict] = {}
    for model in sub["model_name"].unique():
        quality_cache[model] = _load_model_quality(model)

    sub["psnr_mean"] = sub["model_name"].map(lambda m: quality_cache[m]["psnr"])
    sub["bpp_q"] = sub["model_name"].map(lambda m: quality_cache[m]["bpp"])
    # Fall back to BPP from benchmark JSON if quality file unavailable
    sub["bpp_q"] = sub["bpp_q"].fillna(sub["bpp"])
    # Convert energy mJ → J
    sub["energy_per_tile_j"] = sub["energy_per_tile_mj"] / 1000.0

    has_psnr = sub["psnr_mean"].notna().any()

    if not has_psnr:
        print("⚠ No PSNR data found. Showing latency vs BPP scatter instead.")
        x_col = "bpp_q"
        xlabel = "BPP (bits per pixel)"
        x_label_fmt = "{:.4f}"
    else:
        x_col = "psnr_mean"
        xlabel = "PSNR (dB)"
        x_label_fmt = "{:.2f}"

    archs = [a for a in ARCHITECTURE_COLORS if a in sub["architecture"].values]
    plats = [
        p
        for p in PLATFORM_ORDER
        if p in sub["platform"].cat.categories and (sub["platform"] == p).any()
    ]

    fig, (ax_lat, ax_eng) = plt.subplots(1, 2, figsize=(14, 6))

    for ax, y_col, ytitle in [
        (ax_lat, "latency_ms", "Latency (ms)"),
        (ax_eng, "energy_per_tile_j", "Energy per tile (J)"),
    ]:
        for plat in plats:
            for arch in archs:
                grp = sub[(sub["platform"] == plat) & (sub["architecture"] == arch)]
                if grp.empty:
                    continue
                x_vals = grp[x_col].values.astype(float)
                y_vals = grp[y_col].values.astype(float)
                bpp_v = grp["bpp_q"].fillna(0.05).values.astype(float)
                sizes = np.clip(bpp_v * 500, 30, 300)

                ax.scatter(
                    x_vals,
                    y_vals,
                    s=sizes,
                    color=PLATFORM_COLORS.get(plat, "gray"),
                    marker=ARCH_MARKERS.get(arch, "o"),
                    alpha=0.85,
                    edgecolors="white",
                    linewidths=0.8,
                    label=f"{PLATFORM_LABELS.get(plat, plat)} / {arch}",
                    zorder=5,
                )
                # Annotate with lambda
                for xi, yi, lmbda_v, bpp_v_i in zip(x_vals, y_vals, grp["lmbda"].values, bpp_v):
                    bpp_str = f"{bpp_v_i:.3f}" if not np.isnan(bpp_v_i) else ""
                    label = f"λ={lmbda_v}"
                    if bpp_str:
                        label += f"\n({bpp_str} bpp)"
                    ax.annotate(
                        label,
                        (xi, yi),
                        textcoords="offset points",
                        xytext=(5, 5),
                        fontsize=7,
                        color=PLATFORM_COLORS.get(plat, "gray"),
                    )

        ax.set_xlabel(xlabel)
        ax.set_ylabel(ytitle)
        ax.set_title(f"{ytitle} vs {xlabel} — {SCENARIO_LABELS.get(scenario, scenario)}")

        if y_col == "energy_per_tile_j":
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

        # Deduplicated legend
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax.legend(by_label.values(), by_label.keys(), fontsize=8)

    # BPP size legend
    for ax in (ax_lat, ax_eng):
        for bpp_ex in [0.1, 0.3, 0.5]:
            ax.scatter([], [], s=bpp_ex, color="gray", alpha=0.5, label=f"BPP = {bpp_ex}")

    fig.suptitle(f"Hardware Cost Scatter — {scenario} scenario", fontsize=13)
    fig.tight_layout()
    if SAVE_FIGURES:
        PLOTS_DIR.mkdir(parents=True, exist_ok=True)
        plt.savefig(
            PLOTS_DIR / f"hardware_cost_scatter_{scenario}.{FIGURE_FORMAT}", bbox_inches="tight"
        )
    plt.show()


plot_hardware_cost_scatter(df_all, scenario="full")
plot_hardware_cost_scatter(df_all, scenario="compress")


## 7 · Export Summary CSV

Aggregate across seeds: mean ± std for key metrics, grouped by `(architecture, lmbda, platform, scenario)`.

In [ ]:
SUMMARY_METRICS = [
    "latency_ms",
    "nn_latency_ms",
    "cpu_latency_ms",
    "nn_fraction",
    "throughput_fps",
    "power_total_w",
    "energy_per_tile_mj",
    "bpp",
]

agg_funcs = {m: ["mean", "std"] for m in SUMMARY_METRICS if m in df_all.columns}
summary = (
    df_all.groupby(["architecture", "lmbda", "activation", "platform", "scenario"])
    .agg(agg_funcs)
    .round(4)
)
# Flatten MultiIndex columns: (metric, stat) → metric_mean / metric_std
summary.columns = ["_".join(c) for c in summary.columns]
summary = summary.reset_index()

print(f"Summary shape: {summary.shape}")
print(summary[summary["scenario"] == "full"].to_string(float_format="{:.3f}".format, max_rows=30))

if SAVE_FIGURES:
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    out_path = PLOTS_DIR / "hardware_model_comparison_summary.csv"
    summary.to_csv(out_path, index=False)
    print(f"\n✓ Saved: {out_path}")
else:
    print("\nSet SAVE_FIGURES = True to export CSV and PDF figures.")
